# Test extraction on the real PDF

In [3]:
import sys, os
from pathlib import Path
import logging

def find_project_root(marker="backend", start=None):
    current = Path(start or os.getcwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / marker).is_dir():
            return candidate
    raise RuntimeError(f"Could not find a '{marker}' folder above {current}")

PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "backend"))
sys.path.insert(0, str(PROJECT_ROOT / "backend" / "src"))

logging.basicConfig(level=logging.INFO, format="%(message)s")

import io
import pdfplumber

MAX_FILE_SIZE_BYTES = 20 * 1024 * 1024

with open(PROJECT_ROOT / "data" / "Diabetes_Mellitus_Type_2.pdf", "rb") as f:
    file_bytes = f.read()
print(f"File size: {len(file_bytes) / 1024:.1f} KB (limit: {MAX_FILE_SIZE_BYTES / (1024*1024):.0f} MB)")

with pdfplumber.open(io.BytesIO(file_bytes)) as pdf:
    print(f"Pages: {len(pdf.pages)}")
    pages_text = [page.extract_text() or "" for page in pdf.pages]

full_text = "\n\n".join(t for t in pages_text if t.strip())

print(f"\nExtracted {len(full_text)} characters")
print("---")
print(full_text)

File size: 55.2 KB (limit: 20 MB)
Pages: 8

Extracted 10469 characters
---
Diabetes Education – #2
Diabetes Mellitus Type 2
What is it?
Diabetes is a common health problem in the U.S. and the world. In diabetes,
the body does not use the food it digests well. It is hard for the body to use
carbohydrates and fats. The main marker of diabetes is high blood sugar
(“glucose”). Your blood sugar is kept in check by insulin. Insulin is a
hormone that is made in the pancreas. When you get diabetes, it is related to
two things:
• The amount of insulin your body makes
• How well your body’s cells use insulin.
There are two different types of diabetes: type 1 and type 2. Only about 5%
of people have type 1. Type 1 used to be called other names (“juvenile
diabetes”, “insulin-dependent diabetes”). In type 1, the pancreas does not
make insulin. It usually starts as a child or teen. Type 2 often starts after age
40. Type 2 used to be called other names too (“adult-onset diabetes”).
Obese teens can al

# Chunk the extracted text, with multi-document support

In [4]:
from medrag.processing.chunker import sentence_based_chunk
from medrag.processing.models import Chunk
import uuid

def chunk_user_upload(text: str, session_id: str, document_id: str, filename: str, target_tokens: int = 300) -> list:
    """Chunk a user-uploaded document's extracted text. Tagged with
    BOTH session_id (so retrieval can filter to only this user's
    session) and document_id (so a session with multiple uploads can
    distinguish which chunk came from which file, and citations can
    say 'from your uploaded file: X.pdf'). Reuses sentence_based_chunk
    (Phase 5) directly - it's already source-agnostic; only the
    tagging/metadata wrapping here is new."""
    raw_texts = sentence_based_chunk(text, target_tokens=target_tokens)
    chunks = []
    for i, raw_text in enumerate(raw_texts):
        chunk_id = f"{document_id}_upload_{i}"
        chunks.append(Chunk(
            chunk_id=chunk_id,
            point_id=Chunk.make_point_id(chunk_id),
            text=f"{filename}: {raw_text}",
            raw_text=raw_text,
            source="user_upload",
            topics=[],
            source_id=document_id,
            chunk_index=i,
            chunk_type="text",
            metadata={"filename": filename, "session_id": session_id, "document_id": document_id},
        ))
    return chunks


document_id = str(uuid.uuid4())
test_session_id = "test-session-123"

chunks = chunk_user_upload(full_text, test_session_id, document_id, "Diabetes_Mellitus_Type_2.pdf")

print(f"Created {len(chunks)} chunks")
for c in chunks[:3]:
    print(f"\nchunk_id={c.chunk_id}")
    print(f"  text: {c.text[:150]}")

Created 11 chunks

chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_0
  text: Diabetes_Mellitus_Type_2.pdf: Diabetes Education – #2
Diabetes Mellitus Type 2
What is it? Diabetes is a common health problem in the U.S. and the wor

chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_1
  text: Diabetes_Mellitus_Type_2.pdf: At first, your body will
make more insulin to try to keep up. But, when the body can no longer keep
up, diabetes comes o

chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_2
  text: Diabetes_Mellitus_Type_2.pdf: • Damage to the nerves (called “neuropathy”) can cause numbness,
tingling and pain in your feet, legs, and arms. Usually


# Embed and upsert with session tagging

In [6]:
import openai
from fastembed import SparseTextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.http import models as qmodels
from config.settings import settings
from medrag.embeddings.qdrant_client import get_qdrant_client, TEXT_COLLECTION, DENSE_VECTOR_NAME, SPARSE_VECTOR_NAME

openai_client = openai.OpenAI(api_key=settings.openai_api_key)
sparse_model = SparseTextEmbedding(model_name="Qdrant/bm25")
qdrant = get_qdrant_client(settings.qdrant_url or "http://localhost:6333")

def embed_and_upsert_upload_chunks(chunks: list, qdrant_client, openai_client, sparse_model):
    texts = [c.text for c in chunks]

    dense_response = openai_client.embeddings.create(model="text-embedding-3-small", input=texts)
    dense_vectors = [d.embedding for d in dense_response.data]

    sparse_vectors = list(sparse_model.embed(texts))

    points = []
    for chunk, dense_vec, sparse_vec in zip(chunks, dense_vectors, sparse_vectors):
        payload = {
            "chunk_id": chunk.chunk_id,
            "source": chunk.source,
            "topics": chunk.topics,
            "source_id": chunk.source_id,
            "chunk_type": chunk.chunk_type,
            "chunk_index": chunk.chunk_index,
            "text": chunk.text,
            "raw_text": chunk.raw_text,
            "metadata": chunk.metadata,
            "linked_images": [],
            "session_id": chunk.metadata["session_id"],  # the field that makes this session-scoped
            "document_id": chunk.metadata["document_id"],
        }
        points.append(qmodels.PointStruct(
            id=chunk.point_id,
            vector={DENSE_VECTOR_NAME: dense_vec, SPARSE_VECTOR_NAME: qmodels.SparseVector(
                indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist(),
            )},
            payload=payload,
        ))

    qdrant_client.upsert(collection_name=TEXT_COLLECTION, points=points)
    return len(points)


count = embed_and_upsert_upload_chunks(chunks, qdrant, openai_client, sparse_model)
print(f"Upserted {count} chunks")

# Verify total count grew by exactly 11, nothing else disturbed
total = qdrant.count(collection_name=TEXT_COLLECTION, exact=True).count
print(f"medrag_text total count: {total} (expected 22696 + 11 = 22707)")

2026-09-22 16:06:32.272 | WARNING  | fastembed.common.model_management:download_files_from_huggingface:223 - Local file sizes do not match the metadata.
HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
HTTP Request: PUT http://localhost:6333/collections/medrag_text/points?wait=true "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/medrag_text/points/count "HTTP/1.1 200 OK"


Upserted 11 chunks
medrag_text total count: 22707 (expected 22696 + 11 = 22707)


# Test session-scoped retrieval

In [7]:
def search_with_session_filter(qdrant_client, query_text, session_id, openai_client, sparse_model, limit=10):
    dense_vec = openai_client.embeddings.create(model="text-embedding-3-small", input=query_text).data[0].embedding
    sparse_vec = list(sparse_model.embed([query_text]))[0]

    session_filter = qmodels.Filter(
        should=[
            qmodels.IsEmptyCondition(is_empty=qmodels.PayloadField(key="session_id")),
            qmodels.FieldCondition(key="session_id", match=qmodels.MatchValue(value=session_id)),
        ]
    )

    dense_results = qdrant_client.query_points(
        collection_name=TEXT_COLLECTION, query=dense_vec, using=DENSE_VECTOR_NAME,
        query_filter=session_filter, limit=limit,
    ).points

    return dense_results


# Test 1: query using the fictional/unique term from OUR uploaded PDF, correct session
results_correct_session = search_with_session_filter(
    qdrant, "What happens when the body can no longer keep up with insulin production?",
    test_session_id, openai_client, sparse_model, limit=5,
)
print("=== Correct session ===")
for r in results_correct_session:
    print(f"  source={r.payload['source']}  session_id={r.payload.get('session_id')}  chunk_id={r.payload['chunk_id']}")

# Test 2: SAME query, but a DIFFERENT (wrong) session_id - the uploaded chunks must NOT appear
results_wrong_session = search_with_session_filter(
    qdrant, "What happens when the body can no longer keep up with insulin production?",
    "some-other-session-999", openai_client, sparse_model, limit=5,
)
print("\n=== Wrong session (should NOT show our upload) ===")
for r in results_wrong_session:
    print(f"  source={r.payload['source']}  session_id={r.payload.get('session_id')}  chunk_id={r.payload['chunk_id']}")

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/medrag_text/points/query "HTTP/1.1 200 OK"


=== Correct session ===
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_1
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_0
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_4
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_9
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_8


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/medrag_text/points/query "HTTP/1.1 200 OK"



=== Wrong session (should NOT show our upload) ===
  source=openfda  session_id=None  chunk_id=Glimepiride_openfda_5
  source=who  session_id=None  chunk_id=coronary artery disease+heart failure+hyperlipidemia+stroke_who_text_114
  source=who  session_id=None  chunk_id=asthma+copd_who_table_88
  source=openfda  session_id=None  chunk_id=Glipizide_openfda_11
  source=openfda  session_id=None  chunk_id=DAPAGLIFLOZIN_openfda_22


# Mixed-source query in the correct session

In [8]:
results_mixed = search_with_session_filter(
    qdrant, "What are the symptoms and treatment options for type 2 diabetes?",
    test_session_id, openai_client, sparse_model, limit=8,
)
print("=== Mixed query, correct session (should show BOTH sources) ===")
for r in results_mixed:
    print(f"  source={r.payload['source']}  session_id={r.payload.get('session_id')}  chunk_id={r.payload['chunk_id']}")

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/medrag_text/points/query "HTTP/1.1 200 OK"


=== Mixed query, correct session (should show BOTH sources) ===
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_4
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_1
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_9
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_0
  source=who  session_id=None  chunk_id=asthma+copd_who_table_88
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_8
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_3
  source=user_upload  session_id=test-session-123  chunk_id=64c874ad-bd32-4c39-864f-5dce9c3a4276_upload_5


# Delete old test points, redefine with user_id

In [9]:
# Clean up the 11 test points tagged with the old session-only design
qdrant.delete(
    collection_name=TEXT_COLLECTION,
    points_selector=qmodels.FilterSelector(
        filter=qmodels.Filter(
            must=[qmodels.FieldCondition(key="document_id", match=qmodels.MatchValue(value=document_id))]
        )
    ),
)
total_after_delete = qdrant.count(collection_name=TEXT_COLLECTION, exact=True).count
print(f"After cleanup: {total_after_delete} (expected 22696)")

HTTP Request: POST http://localhost:6333/collections/medrag_text/points/delete?wait=true "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/medrag_text/points/count "HTTP/1.1 200 OK"


After cleanup: 22696 (expected 22696)


# Redefine chunking/upsert with user_id, re-test

In [11]:
def chunk_user_upload(text: str, user_id: str, session_id: str, document_id: str, filename: str, target_tokens: int = 300) -> list:
    """Chunk a user-uploaded document's extracted text. Tagged with
    user_id (the actual retrieval-isolation boundary - any chat
    belonging to this user can see this upload), session_id (which
    specific chat the upload happened in, kept for citation display -
    e.g. 'uploaded in this conversation' vs 'uploaded earlier'), and
    document_id (distinguishes multiple uploads from each other)."""
    raw_texts = sentence_based_chunk(text, target_tokens=target_tokens)
    chunks = []
    for i, raw_text in enumerate(raw_texts):
        chunk_id = f"{document_id}_upload_{i}"
        chunks.append(Chunk(
            chunk_id=chunk_id,
            point_id=Chunk.make_point_id(chunk_id),
            text=f"{filename}: {raw_text}",
            raw_text=raw_text,
            source="user_upload",
            topics=[],
            source_id=document_id,
            chunk_index=i,
            chunk_type="text",
            metadata={
                "filename": filename,
                "user_id": user_id,
                "session_id": session_id,
                "document_id": document_id,
            },
        ))
    return chunks


def embed_and_upsert_upload_chunks(chunks: list, qdrant_client, openai_client, sparse_model):
    texts = [c.text for c in chunks]
    dense_response = openai_client.embeddings.create(model="text-embedding-3-small", input=texts)
    dense_vectors = [d.embedding for d in dense_response.data]
    sparse_vectors = list(sparse_model.embed(texts))

    points = []
    for chunk, dense_vec, sparse_vec in zip(chunks, dense_vectors, sparse_vectors):
        payload = {
            "chunk_id": chunk.chunk_id,
            "source": chunk.source,
            "topics": chunk.topics,
            "source_id": chunk.source_id,
            "chunk_type": chunk.chunk_type,
            "chunk_index": chunk.chunk_index,
            "text": chunk.text,
            "raw_text": chunk.raw_text,
            "metadata": chunk.metadata,
            "linked_images": [],
            "user_id": chunk.metadata["user_id"],
            "session_id": chunk.metadata["session_id"],
            "document_id": chunk.metadata["document_id"],
        }
        points.append(qmodels.PointStruct(
            id=chunk.point_id,
            vector={DENSE_VECTOR_NAME: dense_vec, SPARSE_VECTOR_NAME: qmodels.SparseVector(
                indices=sparse_vec.indices.tolist(), values=sparse_vec.values.tolist(),
            )},
            payload=payload,
        ))

    qdrant_client.upsert(collection_name=TEXT_COLLECTION, points=points)
    return len(points)


def search_with_user_filter(qdrant_client, query_text, user_id, openai_client, sparse_model, limit=10):
    """The real isolation boundary: filters by user_id, not session_id -
    any chat belonging to this user can see any of that user's uploads."""
    dense_vec = openai_client.embeddings.create(model="text-embedding-3-small", input=query_text).data[0].embedding

    user_filter = qmodels.Filter(
        should=[
            qmodels.IsEmptyCondition(is_empty=qmodels.PayloadField(key="user_id")),
            qmodels.FieldCondition(key="user_id", match=qmodels.MatchValue(value=user_id)),
        ]
    )

    return qdrant_client.query_points(
        collection_name=TEXT_COLLECTION, query=dense_vec, using=DENSE_VECTOR_NAME,
        query_filter=user_filter, limit=limit,
    ).points


# Re-upload the same document, now tagged to a user with TWO different sessions (simulating "two open chats")
test_user_id = "test-user-abc"
session_a = "session-a-first-chat"
session_b = "session-b-second-chat"

chunks = chunk_user_upload(full_text, test_user_id, session_a, document_id, "Diabetes_Mellitus_Type_2.pdf")
count = embed_and_upsert_upload_chunks(chunks, qdrant, openai_client, sparse_model)
print(f"Uploaded {count} chunks under user_id={test_user_id}, session_id={session_a}")

# Test: query from session_b (a DIFFERENT chat, SAME user) - should still see the upload
results_same_user_diff_session = search_with_user_filter(
    qdrant, "What happens when the body can no longer keep up with insulin production?",
    test_user_id, openai_client, sparse_model, limit=5,
)
print("\n=== Same user, DIFFERENT session (should see upload) ===")
for r in results_same_user_diff_session:
    print(f"  source={r.payload['source']}  user_id={r.payload.get('user_id')}  session_id={r.payload.get('session_id')}")

# Test: query from a completely different user - should NOT see it
results_diff_user = search_with_user_filter(
    qdrant, "What happens when the body can no longer keep up with insulin production?",
    "some-other-user-999", openai_client, sparse_model, limit=5,
)
print("\n=== DIFFERENT user (should NOT see upload) ===")
for r in results_diff_user:
    print(f"  source={r.payload['source']}  user_id={r.payload.get('user_id')}  session_id={r.payload.get('session_id')}")

HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: PUT http://localhost:6333/collections/medrag_text/points?wait=true "HTTP/1.1 200 OK"


Uploaded 11 chunks under user_id=test-user-abc, session_id=session-a-first-chat


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/medrag_text/points/query "HTTP/1.1 200 OK"



=== Same user, DIFFERENT session (should see upload) ===
  source=user_upload  user_id=test-user-abc  session_id=session-a-first-chat
  source=user_upload  user_id=test-user-abc  session_id=session-a-first-chat
  source=user_upload  user_id=test-user-abc  session_id=session-a-first-chat
  source=user_upload  user_id=test-user-abc  session_id=session-a-first-chat
  source=user_upload  user_id=test-user-abc  session_id=session-a-first-chat


HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:6333/collections/medrag_text/points/query "HTTP/1.1 200 OK"



=== DIFFERENT user (should NOT see upload) ===
  source=openfda  user_id=None  session_id=None
  source=who  user_id=None  session_id=None
  source=who  user_id=None  session_id=None
  source=openfda  user_id=None  session_id=None
  source=openfda  user_id=None  session_id=None
